# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahadumar/flyrank-MLinternship-Assignment-1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"CSV not found at {DATA_PATH} — check your working directory: {os.getcwd()}"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Working directory: {os.getcwd()}")
print("Ready.")

Loaded: 30,000 rows, 44 columns
Working directory: /content/flyrank-ml-internship-starter
Ready.


## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2)**

I am choosing Lane 2 because my Week 2 work produced a natural baseline that most
interns don't have: a hand-written expert rule that achieves Precision@20 = 0.900
on a stale-and-visible label. That rule — `score = stale × visible × impressions_90d`,
where `stale = days_since_last_update ≥ 180` and `visible = impressions_90d ≥ 500` —
is a meaningful benchmark, not a throwaway comparison.

The core question this capstone will answer: **can a data-driven scoring model match
or exceed that precision, generalize beyond two hard thresholds, and produce ranked
recommendations with interpretable reason codes?**

The hand rule has a structural limitation this dataset exposes clearly. In this data,
54.2% of pages are declining — but the stale-and-visible filter catches only 17 pages
(0.1% of the dataset). That is not a scale problem; it is a signal problem. The rule's
two hard thresholds miss the vast majority of pages that are already in decline. A
scoring model that combines staleness, position, CTR relative to peers, content age,
and engagement rate can surface the much larger population of refresh candidates that
a two-threshold rule structurally misses.

This lane connects a real operational decision — where does a content team spend
refresh effort next week — to a measurable, honest output: a ranked page list with
scores and reason codes. The Week 2 hand rule is not discarded; it becomes the
capstone's baseline. The research question is whether ML closes the gap between what
the rule flags and what actually needs attention.

In [6]:
# Supporting numbers for Lane 2 choice

# How many pages are stale by the Week 2 definition?
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
stale_and_visible = stale & visible

total = len(df)
n_stale = stale.sum()
n_stale_visible = stale_and_visible.sum()

print(f"Total pages in dataset:               {total:,}")
print(f"Pages stale (≥180 days):              {n_stale:,}  ({n_stale/total*100:.1f}% of all pages)")
print(f"Pages stale AND visible (≥500 impr):  {n_stale_visible:,}  ({n_stale_visible/total*100:.1f}% of all pages)")
print()
print("At this scale, a hand rule that reviews pages manually is not sustainable.")
print("A scored, ranked list is the only practical path to prioritizing refresh effort.")

Total pages in dataset:               30,000
Pages stale (≥180 days):              174  (0.6% of all pages)
Pages stale AND visible (≥500 impr):  17  (0.1% of all pages)

At this scale, a hand rule that reviews pages manually is not sustainable.
A scored, ranked list is the only practical path to prioritizing refresh effort.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages should be prioritized for content refresh, and can
a data-driven scoring model rank them more reliably and generalizably than a
hand-crafted expert rule?

**Decision this improves:** A content strategist or editor deciding where to allocate
writer time in the coming week. Without a principled ranking, this decision is made by
gut feel or recency bias — pages get refreshed because someone noticed them, not
because the data says they need it most.

**Unit of analysis:** One page, measured over a 90-day observation window.

**Output:** A ranked list of pages, each with a refresh opportunity score and a reason
code — for example: *stale + high-impression*, *declining CTR vs position peers*,
*visible but under-capturing clicks*.

**Who acts on it:** A content team lead or SEO strategist.

**Cost of a wrong recommendation:**

- *False positive* (flagging a page that does not need refresh): Writer time wasted on
healthy content. Opportunity cost — that effort could have gone to a genuinely
declining page.

- *False negative* (missing a page that does need refresh): A declining page goes
unnoticed. Traffic continues to erode. By the time the drop is visible in a dashboard,
recovery is harder and slower.

- *The asymmetry matters:* False negatives are costlier in this domain. A missed
opportunity compounds over time. This means that when tuning the model, recall should
be weighted more heavily than precision at the margin — the model should lean toward
flagging rather than missing.

**Why ML helps over a hand rule:** A hand rule fixes thresholds (180 days, 500
impressions). It cannot adapt to pages that are declining fast but have not crossed
those thresholds yet, or to pages that are technically stale but are stable performers
that do not need review. A learned model can combine multiple signals — staleness,
position, CTR relative to peers, engagement rate, content age — and assign a
continuous score that reflects the real opportunity, not just a binary gate.

In [10]:
# Cost asymmetry: show what false negatives look like in this data

# High-value pages that are stale = the costliest false negatives
high_value_stale = df[stale & (df["impressions_90d"] >= 1000)].copy()

print(f"High-impression stale pages (≥1000 impressions AND ≥180 days old): {len(high_value_stale):,}")
print(f"Median impressions on these pages: {high_value_stale['impressions_90d'].median():,.0f}")
print(f"Median avg_position on these pages: {high_value_stale['avg_position'].median():.1f}")
print()

# What fraction of them are already declining?
already_declining = high_value_stale["trend_direction"].str.lower().eq("down").mean()
print(f"Of these high-value stale pages, {already_declining*100:.1f}% are already trending downward.")
print("High-traffic pages already in decline represent the costliest false negatives.")
print("A model that catches declining pages BEFORE they cross staleness thresholds")
print("prevents these losses earlier than any threshold-based rule can.")

High-impression stale pages (≥1000 impressions AND ≥180 days old): 12
Median impressions on these pages: 6,074
Median avg_position on these pages: 20.8

Of these high-value stale pages, 91.7% are already trending downward.
High-traffic pages already in decline represent the costliest false negatives.
A model that catches declining pages BEFORE they cross staleness thresholds
prevents these losses earlier than any threshold-based rule can.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the dataset justify this lane and show it is worth 7 weeks of work:

**Number 1 — The gap between declining pages and what the hand rule catches:**
54.2% of pages in this dataset are declining, but the Week 2 hand rule (stale ≥ 180
days AND visible ≥ 500 impressions) flags only 17 pages — 0.1% of the dataset. That
is not a scale problem; it is a signal problem. A model that goes beyond two hard
thresholds can surface the real population of refresh candidates that the rule misses.
This gap is the entire motivation for building a scoring model.

**Number 2 — Impression concentration among stale pages:**
The top 10% of stale-and-visible pages account for 61.4% of total impressions in that
group. Even within the small set of pages the hand rule does catch, prioritization
matters enormously — not all flagged pages are equal, and a model that ranks by
opportunity rather than just flagging adds real value.

**Number 3 — CTR variance within position tiers:**
Pages at similar average positions show wide variance in CTR. For example, pages in
the page_1 tier have a mean CTR of 0.35 but a standard deviation of 0.50 — meaning
two pages at the same rank can behave very differently. Position alone does not explain
performance. Content-level signals explain the gap, and that variance is exactly what a
scoring model can learn to exploit.

*(All numbers computed live in the code cell below — nothing is hardcoded.)*

In [8]:
# ── Number 1: Scale of the stale problem ──────────────────────────────────────
print("=" * 60)
print("NUMBER 1 — Scale of the stale problem")
print("=" * 60)

stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
stale_and_visible = stale & visible

print(f"Total pages:                    {len(df):,}")
print(f"Stale AND visible:              {stale_and_visible.sum():,}  ({stale_and_visible.mean()*100:.1f}% of dataset)")
print(f"→ At this scale, manual triage is not feasible. A ranked model is necessary.\n")

# ── Number 2: Impression concentration among stale pages ──────────────────────
print("=" * 60)
print("NUMBER 2 — Impression concentration among stale pages")
print("=" * 60)

stale_pages = df[stale_and_visible].copy()
top_decile_threshold = stale_pages["impressions_90d"].quantile(0.90)
top_decile = stale_pages[stale_pages["impressions_90d"] >= top_decile_threshold]
top_decile_share = top_decile["impressions_90d"].sum() / stale_pages["impressions_90d"].sum() * 100

print(f"Top 10% of stale-and-visible pages (≥{top_decile_threshold:,.0f} impressions)")
print(f"account for {top_decile_share:.1f}% of total impressions among stale pages.")
print(f"→ Prioritization matters. A model that ranks by opportunity beats a flat flag.\n")

# ── Number 3: CTR variance within position tiers ──────────────────────────────
print("=" * 60)
print("NUMBER 3 — CTR variance within position tiers")
print("=" * 60)

visible_pages = df[df["impressions_90d"] >= 100]
ctr_stats = visible_pages.groupby("position_tier", observed=True)["ctr"].agg(["mean", "std", "count"])
print(ctr_stats.round(4).to_string())
print()
print("→ Pages at the same position tier show wide CTR spread.")
print("  Content-level signals explain the gap — that is what a scoring model can learn.")

NUMBER 1 — Scale of the stale problem
Total pages:                    30,000
Stale AND visible:              17  (0.1% of dataset)
→ At this scale, manual triage is not feasible. A ranked model is necessary.

NUMBER 2 — Impression concentration among stale pages
Top 10% of stale-and-visible pages (≥39,218 impressions)
account for 61.4% of total impressions among stale pages.
→ Prioritization matters. A model that ranks by opportunity beats a flat flag.

NUMBER 3 — CTR variance within position tiers
                 mean     std  count
position_tier                       
deep           0.0554  0.1699    879
page_1         0.3548  0.5023   8633
page_3_5       0.1424  0.2290   6058
striking       0.2558  0.3467   5903
top_3          0.3341  0.4875    533

→ Pages at the same position tier show wide CTR spread.
  Content-level signals explain the gap — that is what a scoring model can learn.


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**

- *"Pages flagged by this model were observed to have higher rates of staleness and
impression decline than unflagged pages in the held-out validation set."*

- *"The scoring model identified directional signals — days since update, impression
volume, CTR relative to position peers, engagement rate — that were associated with
refresh opportunity in this dataset."*

- *"This tool is designed as decision support: it surfaces candidates for human review,
not automated action."*

- *"The model was validated on a client-holdout split: pages from a given client appear
in either training or evaluation, never both."*

**What this work will never claim:**

- It does not prove that refreshing a page *causes* ranking improvement. Correlation in
observational data is not causation.

- It does not claim to model, reverse-engineer, or predict Google's ranking algorithm.
The features used are content and engagement signals observable at decision time, not
algorithm internals.

- It does not generalize beyond the conditions of this dataset — industry, time period,
and content type all affect what signals matter.

- It is not a production deployment recommendation; it is a decision-support prototype
evaluated on anonymized data.

**Language I will use throughout this capstone:** *observed, measured, associated with,
directional, decision-support, in this dataset, under these conditions.*

**Language I will never use:** *proves, causes, predicts Google, definitive, guaranteed,
algorithm factor.*

In [9]:
# Scientific discipline check: confirm features are pre-decision observables
# (no future-window data, no outcome proxies)

safe_features = [
    "days_since_last_update",   # content freshness — knowable at decision time
    "impressions_90d",          # trailing 90-day visibility — knowable at decision time
    "ctr",                      # click-through rate — knowable at decision time
    "avg_position",             # average search position — knowable at decision time
    "word_count",               # content length — knowable at decision time
    "content_age_days",         # how old the page is — knowable at decision time
    "engagement_rate",          # user engagement — knowable at decision time
]

leaky_features = [
    "trend_pct",        # LEAKY: this IS the label direction in numeric form
    "trend_direction",  # LEAKY: this IS the label
]

print("SAFE FEATURES (pre-decision observables):")
for f in safe_features:
    available = "✓" if f in df.columns else "✗ NOT IN DATASET"
    print(f"  {available}  {f}")

print()
print("LEAKY FEATURES (excluded — outcome proxies):")
for f in leaky_features:
    available = "PRESENT — will exclude" if f in df.columns else "not present"
    print(f"  ✗  {f}  [{available}]")

print()
print("Leakage check from Week 2 notebook confirms:")
print("  trend_pct ≈ perfect Precision@50 = 1.000 when used as a feature.")
print("  This is because trend_pct IS the outcome in numeric form.")
print("  It will be excluded from all capstone modeling.")

SAFE FEATURES (pre-decision observables):
  ✓  days_since_last_update
  ✓  impressions_90d
  ✓  ctr
  ✓  avg_position
  ✓  word_count
  ✓  content_age_days
  ✓  engagement_rate

LEAKY FEATURES (excluded — outcome proxies):
  ✗  trend_pct  [PRESENT — will exclude]
  ✗  trend_direction  [PRESENT — will exclude]

Leakage check from Week 2 notebook confirms:
  trend_pct ≈ perfect Precision@50 = 1.000 when used as a feature.
  This is because trend_pct IS the outcome in numeric form.
  It will be excluded from all capstone modeling.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit repo URL on the card

---

**Lane locked (provisionally):** Lane 2 — Refresh / Content Opportunity Scoring

**Capstone baseline established:** Hand rule from Week 2  
- Precision@20 = 0.900  
- Precision@50 = 0.680  

**Key finding from this notebook:** The hand rule flags only 0.1% of pages, but 54.2%
of pages are declining. That gap — between what a two-threshold rule catches and what
actually needs attention — is the core ML opportunity this capstone will address.

**Next step:** Week 3 — deeper feature engineering and validation design